## Merged Features (Cross-Table)

Now combining cleaned and feature-enriched tables to create master datasets and derive features that require information from multiple sources.

In [24]:
import pandas as pd

In [25]:
orders = pd.read_csv("../data/feature_engineered/orders_fe.csv")
order_items = pd.read_csv("../data/feature_engineered/order_items_fe.csv")
customers = pd.read_csv("../data/feature_engineered/customers_fe.csv")
sellers = pd.read_csv("../data/feature_engineered/sellers_fe.csv")
products = pd.read_csv("../data/feature_engineered/products_fe.csv")
order_payments = pd.read_csv("../data/feature_engineered/order_payments_fe.csv")
order_reviews = pd.read_csv("../data/feature_engineered/order_reviews_fe.csv")
closed_deals = pd.read_csv("../data/feature_engineered/closed_deals_fe.csv")
marketing_leads = pd.read_csv("../data/feature_engineered/marketing_leads_fe.csv")

product_category_name_translations = pd.read_csv(
    "../data/processed/category_translation_clean.csv"
)
geo_locations = pd.read_csv("../data/processed/geolocation_clean.csv")

Before directly performing a merge, I verify the grains and keys of the tables I have, because this is fundamental to a successful merge. This ensures that the feature engineer tables I previously saved are also read correctly.

In [26]:
tables = {
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "sellers": sellers,
    "products": products,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "closed_deals": closed_deals,
    "marketing_leads": marketing_leads,
    "geo_locations": geo_locations,
    "category_translation": product_category_name_translations
}

for name, df in tables.items():
    print(f"{name}: {df.shape}")

orders: (99441, 20)
order_items: (112650, 16)
customers: (99441, 7)
sellers: (3095, 6)
products: (32951, 12)
order_payments: (103886, 5)
order_reviews: (98673, 10)
closed_deals: (842, 18)
marketing_leads: (8000, 8)
geo_locations: (1000163, 5)
category_translation: (71, 2)


I also check if the keys are truly unique. Apart from that, I don't expect order_items, order_payments, and order_reviews to be unique because they are in different grains.

In [27]:
print("orders order_id unique:", orders["order_id"].is_unique)
print("customers customer_id unique:", customers["customer_id"].is_unique)
print("sellers seller_id unique:", sellers["seller_id"].is_unique)
print("products product_id unique:", products["product_id"].is_unique)
print("marketing_leads mql_id unique:", marketing_leads["mql_id"].is_unique)
print("closed_deals mql_id unique:", closed_deals["mql_id"].is_unique)

orders order_id unique: True
customers customer_id unique: True
sellers seller_id unique: True
products product_id unique: True
marketing_leads mql_id unique: True
closed_deals mql_id unique: True


In [28]:
orders_before = len(orders)
customers_before = len(customers)

print("Orders:", orders_before)
print("Customers:", customers_before)

Orders: 99441
Customers: 99441


In [29]:
print(
    "Orders with customer_id:",
    orders["customer_id"].notna().sum()
)

print(
    "Unique customer_ids in orders:",
    orders["customer_id"].nunique()
)

Orders with customer_id: 99441
Unique customer_ids in orders: 99441


In [30]:
df_master = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

In [31]:
print("Before:", len(orders))
print("After:", len(df_master))

Before: 99441
After: 99441


After the orders and customers tables were merged, the number of columns in the df_master table increased, as expected.

In [32]:
print(df_master.shape)

(99441, 26)


Now, I will also bind the necessary properties related to the "order_items" table to this df_master variable. If I used a different variable name for each merge operation, there would be many tables and it would look very messy. So I continue via "df_master".
First, I examine the order_items table.

In [33]:
order_items.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'item_total_cost',
       'freight_ratio', 'items_in_order', 'order_total_price',
       'order_total_freight', 'order_total_cost', 'sellers_in_order',
       'is_free_shipping', 'average_item_price'],
      dtype='str')

But in this case, directly merging the `order_items` table with `df_master` is an incorrect approach. If an order has 3 items, the row in `df_master` will be repeated 3 times. This results in order-level information, such as customer information and order details, being repeated row by row. This corrupts the `grain` of `df_master` and causes the "1 row = 1 order" property to be lost.

In [34]:
order_items.shape

(112650, 16)

Therefore, instead of directly linking the `order_items` table, I'm creating an `order_item_summary` variable. I'm also creating some of the features already present in the `order_items` table here. This is because I want to convert the `item-grain` data into an `order-grain` summary table. This also reduces the number of rows.

In [38]:
order_item_summary = (
    order_items
    .groupby("order_id")
    .agg(
        items_in_order=("order_item_id", "count"),
        order_total_price=("price", "sum"),
        order_total_freight=("freight_value", "sum"),
        order_total_cost=("item_total_cost", "sum"),
        sellers_in_order=("seller_id", "nunique"),
        is_free_shipping=("freight_value", lambda x: (x == 0).all()),
        average_item_price=("price", "mean")
    )
    .reset_index()
)

In [39]:
print(order_item_summary.shape)
print(order_item_summary["order_id"].nunique())

(98666, 8)
98666


After the merge operation, the properties from the order_item_summary table are also added to df_master, and the number of columns naturally increases.

In [41]:
df_master = df_master.merge(
    order_item_summary,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print(df_master.shape)
print(df_master["order_id"].nunique())


(99441, 33)
99441


After merging the `order_item_summary` with `df_master`, I checked the newly added order-level features for missing values.

Since `df_master` was merged using a **left join**, these missing values indicate that 775 orders in `df_master` do not have a matching record in `order_item_summary`.

This is important to investigate later, but the missing values should not be filled with arbitrary values at this stage. First, the remaining order-level tables will be integrated and the reasons for these missing records will be evaluated.

In [42]:
df_master[
    [
        "items_in_order",
        "order_total_price",
        "order_total_freight",
        "order_total_cost",
        "sellers_in_order",
        "is_free_shipping",
        "average_item_price"
    ]
].isna().sum()

items_in_order         775
order_total_price      775
order_total_freight    775
order_total_cost       775
sellers_in_order       775
is_free_shipping       775
average_item_price     775
dtype: int64

Similarly, I'm examining the order_payments table. Looking at the row counts and the number of unique order_id entries, I see that there can be multiple payment records for a single order. In other words, the grain of this table is "1 row = 1 payment record". Again, I need to make this table compliant with df_master.

In [43]:
print(order_payments.shape)
print(order_payments["order_id"].nunique())

(103886, 5)
99440


In [49]:
payment_summary = (
    order_payments
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_count=("payment_sequential", "count")
    )
    .reset_index()
)

In [50]:
print(payment_summary.shape)
print(payment_summary["order_id"].nunique())

(99440, 3)
99440


In [51]:
payment_summary.head()

,order_id,total_payment_value,payment_count
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,1
2,000229ec398224ef6ca0657da4fc703e,216.87,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1
